# 04 · Authoritative Research & Reproducible Analytics

Configurable official-source HTTP fetcher with SSRF controls; SEC XBRL adapter; deterministic business calculations with provenance. Live network access is optional and never simulated as real.

In [ ]:
%run 03_evidence_recommendations.ipynb

In [ ]:
from urllib.parse import urlparse
from html.parser import HTMLParser
import ipaddress, socket
import pandas as pd
import httpx

OFFICIAL_DOMAINS={'sec.gov','cisco.com','agilent.com','ibm.com','google.com','cloud.google.com','fao.org','who.int'}
class HTMLText(HTMLParser):
    def __init__(self):super().__init__();self.parts=[];self.hidden=0
    def handle_starttag(self,tag,attrs):
        if tag in ('script','style','noscript'):self.hidden+=1
    def handle_endtag(self,tag):
        if tag in ('script','style','noscript'):self.hidden=max(0,self.hidden-1)
    def handle_data(self,data):
        if not self.hidden:self.parts.append(data)

def validate_official_url(url):
    p=urlparse(url)
    if p.scheme!='https' or p.port not in (None,443) or p.username or p.password:
        raise ValueError('Only standard HTTPS URLs with no embedded credentials are permitted')
    host=(p.hostname or '').lower().rstrip('.')
    if not any(host==domain or host.endswith('.'+domain) for domain in OFFICIAL_DOMAINS):
        raise ValueError('Domain is not in the approved authoritative-source allowlist')
    # Check resolved addresses: block private/reserved destinations. Fail if DNS cannot be inspected.
    addresses=socket.getaddrinfo(host,443,proto=socket.IPPROTO_TCP)
    if not addresses or any(not ipaddress.ip_address(x[4][0]).is_global for x in addresses):
        raise ValueError('Non-public research host is prohibited')
    return url

def fetch_official_source(url,user_agent=None):
    validate_official_url(url)
    ua=user_agent or os.getenv('RESEARCH_USER_AGENT','Group2HackathonResearch/0.1 academic prototype contact: configure-user-agent')
    with httpx.Client(timeout=20,follow_redirects=False,headers={'User-Agent':ua}) as client:
        with client.stream('GET',url) as response:
            response.raise_for_status()
            if 300<=response.status_code<400:raise ValueError('Redirect rejected; explicitly validate destination')
            ctype=response.headers.get('content-type','').lower()
            if not any(t in ctype for t in ('text/html','text/plain','application/pdf')):
                raise ValueError('Unsupported response Content-Type')
            data=b''
            for part in response.iter_bytes():
                data+=part
                if len(data)>MAX_UPLOAD:raise ValueError('Remote source too large')
    if 'application/pdf' in ctype:
        filename='official-research.pdf'; text_blob=data
    else:
        body=data.decode('utf-8',errors='replace')
        if 'html' in ctype:
            parser=HTMLText();parser.feed(body);body=' '.join(parser.parts)
        filename='official-research.txt'; text_blob=body.encode('utf-8')
    return filename,text_blob

def ingest_official_source(project_id,domain,url):
    filename,blob=fetch_official_source(url)
    output=ingest_blob(project_id,domain,filename,blob,'verified-domain-unverified-content',url)
    with db_connect() as cx:
        cx.execute('INSERT INTO research_sources VALUES (?,?,?,?,?,?,?,?,?,?)',
                   (str(uuid.uuid4()),ensure_project_id(project_id),domain,url,url,urlparse(url).hostname,
                    None,'official-host','retrieved',NOW(),json.dumps({'bytes':len(blob)})))
    return output

def fetch_sec_companyfacts(cik, user_agent=None):
    # Official SEC XBRL API, requires an SEC-compliant identifying User-Agent.
    if not str(cik).isdigit():raise ValueError('CIK must contain digits only')
    ua=user_agent or os.getenv('SEC_USER_AGENT')
    if not ua:raise RuntimeError('Set SEC_USER_AGENT with a real contact name and email')
    url=f'https://data.sec.gov/api/xbrl/companyfacts/CIK{int(cik):010d}.json'
    validate_official_url(url)
    with httpx.Client(timeout=25,follow_redirects=False,headers={'User-Agent':ua,'Accept':'application/json'}) as cl:
        response=cl.get(url);response.raise_for_status()
        if len(response.content)>10*1024*1024:raise ValueError('SEC response too large')
        return response.json()

def _safe_num(v,name):
    try:x=float(v)
    except (TypeError,ValueError):raise ValueError(f'{name} must be numeric')
    if not math.isfinite(x):raise ValueError(f'{name} must be finite')
    return x

def financial_metrics(data,project_id=PROJECT_ID,source_name='synthetic',currency='USD'):
    df=pd.DataFrame(data).copy()
    expected={'year','revenue','cost'}
    if not expected.issubset(df.columns):raise ValueError('Required: year, revenue, cost')
    if len(df)<2:raise ValueError('Need at least two years')
    if df['year'].duplicated().any():raise ValueError('Duplicate reporting years')
    df['year']=pd.to_numeric(df['year'],errors='raise').astype(int)
    for col in ['revenue','cost']:
        df[col]=pd.to_numeric(df[col],errors='raise')
        if not np.isfinite(df[col].to_numpy(dtype=float)).all():raise ValueError('Non-finite financial data')
    if (df['revenue']<=0).any() or (df['cost']<0).any():
        raise ValueError('Revenue must be positive and costs nonnegative')
    df=df.sort_values('year')
    years=df['year'].to_numpy()
    if any((b-a)!=1 for a,b in zip(years,years[1:])):raise ValueError('Annual observations must be consecutive')
    df['profit']=df['revenue']-df['cost']
    df['margin_pct']=100*df['profit']/df['revenue']
    df['revenue_growth_pct']=100*df['revenue'].pct_change()
    horizon=int(years[-1]-years[0]);start=float(df['revenue'].iloc[0]);end=float(df['revenue'].iloc[-1])
    cagr=100*((end/start)**(1/horizon)-1)
    result={'source':source_name,'currency':currency,'cagr_pct':round(cagr,4),
       'latest_margin_pct':round(float(df['margin_pct'].iloc[-1]),4),
       'rows':json.loads(df.replace({np.nan:None}).to_json(orient='records')),
       'formulas':{'margin_pct':'100 * (revenue - cost)/revenue',
                   'revenue_growth_pct':'100 * (revenue_t/revenue_t_minus_1 - 1)',
                   'cagr_pct':'100 * ((last_revenue/first_revenue) ** (1/years_elapsed) - 1)'},
       'limitations':['Descriptive results only. No causal inference or market forecast.']}
    data_hash=hashlib.sha256(df.to_json(orient='records').encode()).hexdigest()
    run_id=str(uuid.uuid4())
    with db_connect() as cx:
        cx.execute('INSERT INTO analysis_runs VALUES (?,?,?,?,?,?)',
                   (run_id,ensure_project_id(project_id),'financial_metrics_v1',data_hash,json.dumps(result),NOW()))
    return {'analysis_id':run_id,**result}

def revenue_scenarios(base_revenue,base_margin_pct,changes=(-0.1,0,0.1)):
    rev=_safe_num(base_revenue,'base_revenue');margin=_safe_num(base_margin_pct,'base_margin_pct')
    if rev<=0 or not 0<=margin<=100:raise ValueError('Invalid revenue or margin')
    output=[]
    for change in changes:
        d=_safe_num(change,'scenario change');new=rev*(1+d)
        output.append({'revenue_change_pct':100*d,'projected_revenue':round(new,2),
                       'projected_profit':round(new*margin/100,2),
                       'assumption':'Constant margin; scenario, not a forecast'})
    return output

print('Research adapters and financial metrics ready')

## Batch research task execution

Processes configured official URLs, or existing indexed evidence. The explicit `synthetic_fixture` mode never masquerades as live research. Results preserve evidence identifiers, citations, and per-source failure status. Synchronous execution is appropriate only for the hackathon demo; the shared worker will own long-running jobs later.

In [ ]:
def run_research_task(project_id,domain,question,source_mode='indexed',urls=None,top_k=5):
    project_id=ensure_project_id(project_id)
    if domain not in DOMAINS:raise ValueError('Unknown research domain')
    if not question or len(question)<5:raise ValueError('Research question is too short')
    if source_mode not in {'indexed','official_urls','synthetic_fixture'}:
        raise ValueError('Invalid source_mode')
    urls=list(urls or [])
    if source_mode=='official_urls' and not urls:raise ValueError('Official URLs are required')
    if len(urls)>8:raise ValueError('At most eight approved URLs per research task')
    if source_mode!='official_urls' and urls:raise ValueError('URLs may only be used with official_urls mode')
    task_id=str(uuid.uuid4());started=NOW()
    with db_connect() as cx:
        cx.execute('INSERT INTO research_tasks VALUES (?,?,?,?,?,?,?, ?,?)',
                   (task_id,project_id,domain,question,source_mode,'running','{}',started,None))
    source_results=[]
    if source_mode=='official_urls':
        for url in urls:
            try:
                record=ingest_official_source(project_id,domain,url)
                source_results.append({'url':url,'status':'indexed','document_id':record['document_id']})
            except Exception as err:
                source_results.append({'url':url,'status':'failed','error':type(err).__name__+': '+str(err)[:180]})
    if source_mode=='official_urls':
        success={x['url'] for x in source_results if x['status']=='indexed'}
        hits=hybrid_search(project_id,question,domain,top_k=top_k,source_urls=list(success))
    elif source_mode=='synthetic_fixture':
        hits=hybrid_search(project_id,question,domain,top_k=top_k,source_type='SYNTHETIC')
    else:
        hits=hybrid_search(project_id,question,domain,top_k=top_k,exclude_synthetic=True)
    # Require substantive overlap with the question; TF-IDF char n-grams can otherwise
    # surface irrelevant passages that share only fixture boilerplate.
    stop={'how','what','which','where','should','about','from','with','into','does','that',
          'these','those','could','would','reduce','help','improve','synthetic','test','fixture'}
    essential={t for t in tokens(question) if len(t)>3 and t not in stop}
    if essential:
        min_overlap=2 if len(essential)>=3 else 1
        hits=[h for h in hits if len(essential & set(tokens(h['excerpt'])))>=min_overlap]
    evidence_ids=evidence_from_hits(project_id,hits)
    status=('completed' if hits else 'insufficient_evidence')
    output={'research_task_id':task_id,'project_id':project_id,'domain':domain,
            'question':question,'source_mode':source_mode,'status':status,
            'findings':[{'claim_excerpt':h['excerpt'][:700],'citation':h['citation'],
                         'evidence_id':eid,'source_url':h['source_url'],
                         'source_type':h['source_type'],'relevance_score':h['score']}
                        for eid,h in zip(evidence_ids,hits)],
            'sources':source_results,
            'warnings':['No relevant evidence from the selected sources; cannot make grounded claims'] if not hits else []}
    with db_connect() as cx:
        cx.execute('UPDATE research_tasks SET status=?,result_json=?,completed_at=? WHERE id=?',
                   (status,json.dumps(output),NOW(),task_id))
    return output

def retrieve_research_task(project_id,task_id):
    with db_connect() as cx:
        row=cx.execute('SELECT result_json FROM research_tasks WHERE project_id=? AND id=?',
                       (ensure_project_id(project_id),str(uuid.UUID(str(task_id))))).fetchone()
    return json.loads(row['result_json']) if row else None

print('Batch research task runner ready')